In [1]:
# !pip install selenium pandas
# !pip install html5lib


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# all imports for the notebook
import time
import requests
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup, Comment

# selenium imports
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import (
    NoSuchElementException,
    ElementClickInterceptedException,
    TimeoutException,
)
from webdriver_manager.chrome import ChromeDriverManager

# requests retry adapter
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [3]:
# usage:
# season_df, players_df, matches_df = scrape_understat(league: str, year: int)
#   example: season, players, matches = scrape_understat("EPL", 2023)
# output:
#   season_df: season summary table
#   players_df: players statistics table
#   matches_df: all matches across all matchdays

def scrape_understat(league: str, year: int):
    """
    scrape understat league data: season table, players table, matches across all matchdays.

    returns:
        season_table, players_table, matches_table
    """

    # --- setup browser ---
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")  # start maximized
    options.add_experimental_option("excludeSwitches", ["enable-logging"])  # suppress logs

    # use webdriver directly (avoid WinError 193)
    driver = webdriver.Chrome(options=options)

    # --- go to league page ---
    url = f"https://understat.com/league/{league}/{year}"
    print(f"loading {url}")
    driver.get(url)

    # wait for manual user changes if needed (e.g., accept cookies, language)
    print("waiting 30 seconds for manual interaction...")
    time.sleep(30)

    wait = WebDriverWait(driver, 20)  # setup explicit wait

    # --- helper: convert table container to dataframe ---
    def table_to_df(container):
        try:
            table = container.find_element(By.TAG_NAME, "table")  # try to find table inside container
        except NoSuchElementException:
            table = container  # fallback if container is already the table

        rows = table.find_elements(By.TAG_NAME, "tr")
        data = []
        for row in rows:
            cells = row.find_elements(By.TAG_NAME, "th") or row.find_elements(By.TAG_NAME, "td")
            row_text = [c.text.strip() for c in cells]
            if row_text:
                data.append(row_text)

        if len(data) < 2:  # if table is empty or only header
            return pd.DataFrame()

        return pd.DataFrame(data[1:], columns=data[0])  # first row is header

    # --- 1️⃣ season table ---
    print("extracting season table...")
    season_table = pd.DataFrame()
    try:
        tables = driver.find_elements(By.TAG_NAME, "table")  # find all tables
        for table in tables:
            rows = table.find_elements(By.TAG_NAME, "tr")
            if len(rows) > 15:  # heuristic: season table has more than 15 rows
                header = rows[0].text.lower()
                if any(word in header for word in ["team", "points", "squad"]):  # identify table by keywords
                    driver.execute_script("arguments[0].scrollIntoView({behavior:'smooth',block:'center'});", table)
                    time.sleep(1.5)  # small delay for table to render
                    season_table = table_to_df(table)
                    break
    except Exception:
        pass
    print(f"season table rows: {len(season_table)}")

    # --- 2️⃣ players table ---
    print("extracting players table...")
    players_xpath = "/html/body/div[1]/div[3]/div[4]/div"
    try:
        el = driver.find_element(By.XPATH, players_xpath)
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)  # scroll to element
        time.sleep(1.5)
        wait.until(EC.presence_of_element_located((By.XPATH, players_xpath)))  # wait until table loaded
        players_table = table_to_df(el)
    except Exception:
        players_table = pd.DataFrame()
    print(f"players table rows: {len(players_table)}")

    # --- 3️⃣ matches across all matchdays ---
    print("extracting matches across all matchdays...")
    matches_xpath_root = "/html/body/div[1]/div[3]/div[2]/div/div/div[2]"  # container for matchdays
    prev_button_xpath = "/html/body/div[1]/div[3]/div[2]/div/div/button[1]"  # button to go to previous matchday

    # scroll to matchday selector
    try:
        driver.execute_script(
            "arguments[0].scrollIntoView({block:'center'});",
            driver.find_element(By.XPATH, prev_button_xpath)
        )
    except Exception:
        pass
    time.sleep(1)

    all_matchdays = []

    # --- helper: extract matches from current matchday ---
    def extract_matchday():
        try:
            container = driver.find_element(By.XPATH, matches_xpath_root)
            date_blocks = container.find_elements(By.XPATH, "./div")  # each block is a matchday
        except Exception:
            return pd.DataFrame()

        rows = []
        for block in date_blocks:
            try:
                date = block.find_element(By.XPATH, "./div[1]").text.strip()  # extract date
                match_container = block.find_element(By.XPATH, "./div[2]")
                matches = match_container.find_elements(By.XPATH, "./div")  # each match
            except Exception:
                continue

            for m in matches:
                try:
                    text = m.text.strip()
                    if not text:
                        continue
                    lines = [x.strip() for x in text.split("\n") if x.strip()]
                    try:
                        # extract home, away, score using CSS classes
                        home = m.find_element(By.CSS_SELECTOR, ".team-home .team-title").text.strip()
                        away = m.find_element(By.CSS_SELECTOR, ".team-away .team-title").text.strip()
                        score = m.find_element(By.CSS_SELECTOR, ".match-info .score").text.strip()
                    except Exception:
                        # fallback parsing if CSS classes missing
                        home = lines[0]
                        away = lines[-1]
                        score = ""
                        for mid in lines[1:-1]:
                            if any(c.isdigit() for c in mid) and len(mid) <= 3:
                                score = mid
                                break
                    rows.append({
                        "date": date,
                        "home_team": home,
                        "away_team": away,
                        "score": score
                    })
                except Exception:
                    continue
        return pd.DataFrame(rows)

    # loop through all previous matchdays
    while True:
        df = extract_matchday()
        if not df.empty:
            all_matchdays.insert(0, df)  # prepend to list (chronological order)

        try:
            prev_btn = driver.find_element(By.XPATH, prev_button_xpath)
            disabled = driver.execute_script("return arguments[0].disabled;", prev_btn)
            if disabled:  # stop if no previous matchday
                break
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", prev_btn)
            time.sleep(0.5)
            try:
                prev_btn.click()  # try normal click
            except ElementClickInterceptedException:
                driver.execute_script("arguments[0].click();", prev_btn)  # fallback click
            time.sleep(3)  # wait for matchday to load
        except Exception:
            break

    matches_table = pd.concat(all_matchdays, ignore_index=True) if all_matchdays else pd.DataFrame()
    print(f"matches table rows: {len(matches_table)} from {len(all_matchdays)} matchdays")

    driver.quit()  # close browser
    return season_table, players_table, matches_table

In [4]:
# usage:
# out = scrape_transfermarkt(verbose: bool)
#   returns: {"injuries": injuries_df, "balance_vs_position": balance_df}

def scrape_transfermarkt(verbose: bool = True):
    """
    scrape transfermarkt pages for:
    - premier league injuries table
    - transfer balance vs league position table
    """

    # -----------------
    # helper functions
    # -----------------

    def _log(msg):
        if verbose:
            print(msg)

    def _build_session():
        """create a requests session with retry and realistic headers."""
        s = requests.Session()
        s.headers.update({
            "User-Agent": "Mozilla/5.0",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://www.transfermarkt.us/",
        })
        retries = Retry(
            total=5,
            backoff_factor=0.6,
            status_forcelist=[403, 429, 500, 502, 503, 504],
            allowed_methods=["GET"]
        )
        s.mount("https://", HTTPAdapter(max_retries=retries))
        s.mount("http://", HTTPAdapter(max_retries=retries))
        return s

    def _fetch_html(url: str) -> str:
        """try requests, fall back to selenium if blocked."""
        sess = _build_session()
        r = sess.get(url, timeout=20)
        if r.status_code == 200 and r.text.strip():
            return r.text

        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
        driver.get(url)
        time.sleep(2)
        html = driver.page_source
        driver.quit()
        return html

    def _normalize_columns(df):
        """flatten multiindex columns and standardize names."""
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [
                "_".join([c for c in map(str, col) if c and not str(c).startswith("Unnamed")])
                for col in df.columns
            ]
        df.columns = [
            str(c).strip().lower().replace(" ", "_")
            for c in df.columns
        ]
        return df

    def _read_first_table(html):
        """read the first table from html."""
        dfs = pd.read_html(StringIO(html))
        df = dfs[0]
        return _normalize_columns(df)

    # -----------------
    # scrape injuries
    # -----------------

    INJ_URL = "https://www.transfermarkt.us/premier-league/verletztespieler/wettbewerb/GB1"
    BAL_URL = "https://www.transfermarkt.us/premier-league/transferbilanztabellenplatz/wettbewerb/GB1"

    _log("fetching injuries page...")
    injuries_html = _fetch_html(INJ_URL)
    injuries_raw = _read_first_table(injuries_html)

    # basic cleaning logic preserved from your original code
    def _clean_injuries(df):
        """clean raw injuries table into a player-position-club-injury table."""
        df = df.copy()
        df.columns = [c.lower().replace(" ", "_") for c in df.columns]
        # minimal restructuring: many tables align correctly already
        return df

    injuries_df = _clean_injuries(injuries_raw)

    # -----------------
    # scrape balance table
    # -----------------

    _log("fetching balance-vs-position page...")
    balance_html = _fetch_html(BAL_URL)
    balance_raw = _read_first_table(balance_html)

    def _clean_balance(df):
        """clean balance-vs-position table."""
        df = df.copy()
        df.columns = [c.lower().replace(" ", "_") for c in df.columns]
        return df

    balance_df = _clean_balance(balance_raw)

    return {"injuries": injuries_df, "balance_vs_position": balance_df}

In [5]:
# usage:
# tables = scrape_fbref(season_start_year: int)
# example: tables = scrape_fbref(2024)
# returns: dict of all extracted tables {table_name: dataframe}

def scrape_fbref(season_start_year: int):
    """
    scrape fbref premier league stats for the given season.
    extracts all visible and commented tables on the main season page.
    """

    season_str = f"{season_start_year}-{season_start_year + 1}"
    url = f"https://fbref.com/en/comps/9/{season_str}/{season_str}-Premier-League-Stats"

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://fbref.com/",
    }

    print(f"fetching: {url}")
    s = requests.Session()
    s.headers.update({
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0.0.0 Safari/537.36"),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://fbref.com/en/",
        "Connection": "keep-alive",
    })
    
    r = s.get(url, timeout=20)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    tables = {}

    def extract(div):
        """extract a table from a table_container div."""
        table = div.find("table")
        if not table:
            return
        table_id = div.get("id", "unknown").replace("div_", "")

        try:
            df_list = pd.read_html(StringIO(str(table)), header=1)
            df = df_list[0]
        except:
            return

        # flatten multiindex headers
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = ["_".join([str(x) for x in col if x]) for col in df.columns]

        tables[table_id] = df
        print(f"  extracted: {table_id}")

    # extract visible tables
    for div in soup.find_all("div", class_="table_container"):
        extract(div)

    # extract tables hidden inside comments
    for comment in soup.find_all(string=lambda s: isinstance(s, Comment)):
        sub = BeautifulSoup(comment, "html.parser")
        for div in sub.find_all("div", class_="table_container"):
            extract(div)

    print(f"total tables extracted: {len(tables)}")
    return tables

In [7]:
# final test cell to verify all three scrapers work

print("TESTING UNDERSTAT SCRAPER")

# try:
#     season_df, players_df, matches_df = scrape_understat("EPL", 2023)

#     print(f"Understat season table:   {season_df.shape}")
#     print(f"Understat players table:  {players_df.shape}")
#     print(f"Understat matches table:  {matches_df.shape}")
#     print("Understat ✔ Completed\n")

# except Exception as e:
#     print("Understat ❌ Failed:", e)

print("TESTING TRANSFERMARKT SCRAPER")

try:
    tm = scrape_transfermarkt(verbose=False)

    inj = tm.get("injuries")
    bal = tm.get("balance_vs_position")

    print(f"Transfermarkt injuries table:          {inj.shape if inj is not None else None}")
    print(f"Transfermarkt balance-vs-position:     {bal.shape if bal is not None else None}")
    print("Transfermarkt ✔ Completed\n")

except Exception as e:
    print("Transfermarkt ❌ Failed:", e)

print("TESTING FBREF SCRAPER")

try:
    fb_tables = scrape_fbref(2015)

    print(f"FBref tables extracted: {len(fb_tables)}")
    print("Some FBref keys:", list(fb_tables.keys())[:5])
    print("FBref ✔ Completed\n")

except Exception as e:
    print("FBref ❌ Failed:", e)

print("🏁 ALL SCRAPER TESTS FINISHED")

TESTING UNDERSTAT SCRAPER
TESTING TRANSFERMARKT SCRAPER
Transfermarkt injuries table:          (129, 5)
Transfermarkt balance-vs-position:     (20, 7)
Transfermarkt ✔ Completed

TESTING FBREF SCRAPER
fetching: https://fbref.com/en/comps/9/2015-2016/2015-2016-Premier-League-Stats
  extracted: results2015-201691_overall
  extracted: results2015-201691_home_away
  extracted: stats_squads_standard_for
  extracted: stats_squads_standard_against
  extracted: stats_squads_keeper_for
  extracted: stats_squads_keeper_against
  extracted: stats_squads_shooting_for
  extracted: stats_squads_shooting_against
  extracted: stats_squads_playing_time_for
  extracted: stats_squads_playing_time_against
  extracted: stats_squads_misc_for
  extracted: stats_squads_misc_against


C:\Users\soham\AppData\Local\Temp\ipykernel_25344\2206563029.py:65: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  sub = BeautifulSoup(comment, "html.parser")


  extracted: nations
total tables extracted: 13
FBref tables extracted: 13
Some FBref keys: ['results2015-201691_overall', 'results2015-201691_home_away', 'stats_squads_standard_for', 'stats_squads_standard_against', 'stats_squads_keeper_for']
FBref ✔ Completed

🏁 ALL SCRAPER TESTS FINISHED


C:\Users\soham\AppData\Local\Temp\ipykernel_25344\2206563029.py:65: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  sub = BeautifulSoup(comment, "html.parser")


In [24]:
# Project overview — data aggregation for EPL stats (2015-16 to current)

# Official Premier League 3-letter team acronyms (use these for “team” column)
# Example mapping (not exhaustive; expand as needed for all clubs):
# ARS — Arsenal  
# AVL — Aston Villa  
# BOU — Bournemouth  
# BRE — Brentford  
# BRI — Brighton & Hove Albion  
# BUR — Burnley  
# CHE — Chelsea  
# CRY — Crystal Palace  
# EVE — Everton  
# FUL — Fulham  
# LIV — Liverpool  
# MCI — Manchester City  
# MUN — Manchester United  
# NEW — Newcastle United  
# NFO — Nottingham Forest  
# TOT — Tottenham Hotspur  
# WHU — West Ham United  
# WOL — Wolverhampton Wanderers  
# (If new/promoted clubs appear, add their acronym accordingly.)  :contentReference[oaicite:0]{index=0}

# Each data source / contributor has a dedicated function producing a dictionary:  
#   key = season (e.g. "2024-25")  
#   value = pandas.DataFrame with first column "team" (acronym), then one column per stat.

# Missing or unavailable stat values must be filled with the string "NONE" (unique placeholder).

# Output: export your season-by-season DataFrame dictionary to .csv -- one .csv with different tabs for one year (search up how to do this)

# Functions / authors:

# INJE: aggregate_understat()  
#   – uses scrape_understat()  
#   – returns dict[str, DataFrame] for each season  
# ALEX: aggregate_fbref()  
#   – uses scrape_fbref()  
#   – same output format  
# SHIVANTH: aggregate_sofascore() (or similar)  
#   – gather Sofascore + derived stats, same output format  

# Later:  
#   – When combining across sources, rely on “team” column (acronym) to align / merge data  -- Soham will do this

# Example output (after one function):  
# {  
#   "2024-25": DataFrame with columns ["team", "xG", "xA", ...],  
#   "2023-24": DataFrame with same structure, ...  
# }  